# Hesitation: think time and hand hovering as teaching signals

**Deliverable:** input features for the player model (README Teaching: *"handling pressure"*,
*"decision making"*). A physical board gives signals an online platform cannot: how long the
human thinks, and where their hand hovers before committing (a hand that hovers over the queen and
then moves a pawn says something about the decision).

Two sources:

* **Think time** per move: from the moment the robot's move is verified to the moment the human's
  move is seen as stable (`MoveEvent.think_time_s` from `game_manager.ipynb`).
* **Hover events** from vision: `gambit_vision` reports when a hand is over the board and,
  ideally, over which square. Data structures are defined here.

**TODO checklist**
- [ ] Data structure helpers (`HoverEvent.duration`, `MoveTiming` properties).
- [ ] Per-move think-time features and the listener that collects them (`HesitationTracker`).
- [ ] Vision side: hand detection + hand-to-square mapping -> `HoverEvent`s (agree format with gambit_vision).
- [ ] `hesitation_score`: think time relative to the player's own baseline and to position
      complexity (number of legal moves, eval spread of the top multipv lines).
- [ ] Decide which features the player model uses (habits.ipynb) and validate that they predict
      mistakes at all before relying on them.
- [ ] Self-checks (plan at the bottom).

In [ ]:
from __future__ import annotations

import statistics
from dataclasses import dataclass, field
from typing import Optional

import chess

## Data structures

In [ ]:
@dataclass
class HoverEvent:
    t_start: float
    t_end: float
    square: Optional[chess.Square] = None     # None: hand over the board but square unknown
    touched: bool = False                     # piece lifted/touched (if vision can tell)

    @property
    def duration(self) -> float:
        """Length of the hover in seconds (never negative)."""
        # TODO: implement
        raise NotImplementedError


@dataclass
class MoveTiming:
    ply: int
    by: str
    think_time_s: float
    hovers: list = field(default_factory=list)       # HoverEvents during this think
    n_legal_moves: int = 0

    @property
    def hovered_squares(self) -> list:
        """Squares of the hovers that have a known square."""
        # TODO: implement
        raise NotImplementedError

    @property
    def hover_time_s(self) -> float:
        """Total hover duration during this think."""
        # TODO: implement
        raise NotImplementedError


@dataclass
class HesitationFeatures:
    ply: int
    think_time_s: float
    think_ratio_to_median: float      # think time / median human think time so far (1.0 = typical)
    n_hovers: int
    hover_time_s: float
    distinct_hovered_squares: int
    n_legal_moves: int

## Collector

`HesitationTracker` is a game-manager listener (`listeners=[tracker]`). Hover events from vision
are pushed with `add_hover` while the human thinks; they are attached to the next human move.

In [ ]:
class HesitationTracker:
    def __init__(self):
        self.timings: list[MoveTiming] = []
        self._pending_hovers: list[HoverEvent] = []

    def add_hover(self, ev: HoverEvent) -> None:
        """Buffer a hover event until the next human move."""
        # TODO: implement
        raise NotImplementedError

    def __call__(self, event) -> None:          # event: game_manager.MoveEvent
        """Listener: record a MoveTiming for every move (human and robot)."""
        # Design:
        # - n_legal_moves from the position before the move (a crude complexity measure).
        # - Human moves take ownership of the buffered hovers and clear the buffer; robot moves
        #   get none.
        # TODO: implement
        raise NotImplementedError

    def human_timings(self) -> list[MoveTiming]:
        """Only the human's MoveTimings."""
        # TODO: implement
        raise NotImplementedError

    def features(self) -> list[HesitationFeatures]:
        """One HesitationFeatures per human move."""
        # Design:
        # - think_ratio_to_median uses the median of the human's think times *so far* (including
        #   this move), so the feature is causal (could be computed live); guard a zero median.
        # - Hover counts, total hover time and distinct hovered squares from MoveTiming.
        # TODO: implement
        raise NotImplementedError

## Hover extraction and a hesitation score

In [ ]:
def hovers_from_observations(observations: list) -> list[HoverEvent]:
    """Turn a stream of vision observations (BoardObservation with hand_present and, later, a hand
    square) into HoverEvents."""
    # Design hints:
    # - Needs a hand position from gambit_vision (not in BoardObservation yet).
    # - Merge consecutive frames over the same square into one event; drop events shorter than
    #   ~0.2 s (noise).
    # TODO: implement
    raise NotImplementedError


def hesitation_score(features: HesitationFeatures, baseline: dict) -> float:
    """How unusually long/indecisive this move was for this player (0 = typical)."""
    # Design hints:
    # - e.g. z-score of log(think time) against the player's history (`baseline` = mean/std of
    #   log think time per phase), plus a term for hovering over several pieces.
    # Decisions for you: the weights; calibrate on logs.
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
from types import SimpleNamespace

# Self-check plan (write these as asserts once the functions above work):
# - Tracker with two hovers (0..1.5 s on d1, 2..2.5 s on e2), then events (SimpleNamespace with
#   ply, by, fen_before, think_time_s): human 10 s, robot 0.5 s, human 30 s.
# - features() has 2 entries; the first has 2 hovers, 2.0 s hover time, 2 distinct squares and
#   20 legal moves (start position); the second has 0 hovers and ratio 30 / median(10, 30).
# Example shape of a test:
# assert abs(HoverEvent(0.0, 1.5).duration - 1.5) < 1e-9
print("Self-check: not written yet (see plan above)")